In [0]:
import sys
sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/Medalion_Project")

from pyspark.sql.functions import current_timestamp
from datetime import datetime
from utils.config_reader import ConfigReader
from utils.audit_manager import AuditManager

In [0]:
start_time = datetime.now()
# read config file
config=ConfigReader.read_config("/Workspace/Users/nalluriravi3@gmail.com/Medalion_Project/Configs/bronze_config.json")
customer_config=config["customer"]
target_df=spark.table(f"{customer_config['target_catalog']}."
                                f"{customer_config['target_schema']}."
                                f"{customer_config['target_table']}"
                                )


In [0]:

# read source data
from pyspark.sql.functions import col
bronze_df=spark.read.format(customer_config["source_format"]) \
                    .option("header",customer_config["header"])\
                    .option("delimiter",customer_config["delimiter"])\
                    .option("mergeSchema",customer_config["schema_evolution"])\
                    .load(customer_config["source_path"])
bronze_df.display()

# handle renamed column in source. 
# rename_mapping = {
#     "mobile": "contact",
#     "email_id":"email"
# }

rename_mapping=customer_config.get("rename_mapping",{})

bronze_columns = bronze_df.columns


for old_col, new_col in rename_mapping.items():
    if old_col in bronze_columns:
        bronze_df = bronze_df.withColumnRenamed(old_col, new_col)


# Handle dayatype change in source
tgt_types = dict(target_df.dtypes)

# compare and cast source columns
for c, dtype in bronze_df.dtypes:

    if c in tgt_types and dtype != tgt_types[c]:

        bronze_df = bronze_df.withColumn(
            c,
            col(c).cast(tgt_types[c])
        )


In [0]:
# Add Audit Column
bronze_df = bronze_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)
            
# record read count
records_read=bronze_df.count()
try:
    # write to target table
    bronze_df.write.format("delta") \
                .mode("append") \
                .option("mergeSchema",customer_config["schema_evolution"]) \
                .saveAsTable(f"{customer_config['target_catalog']}."
                                f"{customer_config['target_schema']}."
                                f"{customer_config['target_table']}"
                                )
    end_time = datetime.now()

    # Success Audit Logging
    AuditManager.log_audit(
        spark=spark,
        pipeline_name="customer_pipeline",
        layer_name="bronze",
        table_name=customer_config["target_table"],
        start_time=start_time,
        end_time=end_time,
        records_read=records_read,
        records_written=records_read,
        records_rejected=0,
        status="SUCCESS",
        error_message=""
    )
except Exception as e:

    end_time = datetime.now()

    # Failure Audit Logging
    AuditManager.log_audit(
        spark=spark,
        pipeline_name="customer_pipeline",
        layer_name="bronze",
        table_name=customer_config["target_table"],
        start_time=start_time,
        end_time=end_time,
        records_read=0,
        records_written=0,
        records_rejected=0,
        status="FAILED",
        error_message=str(e)
    )
    print(f"Pipeline Failed : {str(e)}")
    raise e




In [0]:
%sql
--ALTER TABLE workspace.bronze.customer_bronze SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
-- alter table workspace.bronze.customer_bronze drop column mobile

-- select * from workspace.bronze.customer_bronze